In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.models import Model
import joblib
from tensorflow.keras.models import load_model
import joblib
import xgboost as xgb
from tensorflow.keras.losses import MeanSquaredError

In [2]:
# Load dataset
df = pd.read_csv('../datasets/perfect_Balance_dataset_cic-ids-2017.csv')

In [3]:
# Drop missing or NaN rows
df.dropna(inplace=True)

In [4]:
# Separate features and labels
X = df.drop(['label', 'label_binary'], axis=1)
y = df['label_binary']

In [5]:
# Encode labels if needed
if y.dtype == 'object':
    y = LabelEncoder().fit_transform(y

SyntaxError: unexpected EOF while parsing (3468599291.py, line 3)

In [6]:
# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split dataset
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [7]:
# Autoencoder
input_dim = X_train.shape[1]
encoding_dim = 14

input_layer = Input(shape=(input_dim,))
encoded = Dense(encoding_dim * 2, activation='relu')(input_layer)
encoded = Dense(encoding_dim, activation='relu')(encoded)

decoded = Dense(encoding_dim * 2, activation='relu')(encoded)
decoded = Dense(input_dim, activation='sigmoid')(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
encoder = Model(inputs=input_layer, outputs=encoded)

autoencoder.compile(optimizer='adam', loss='mse')
autoencoder.fit(X_train, X_train, epochs=10, batch_size=256, validation_data=(X_test, X_test), verbose=1)

Epoch 1/10
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.6819 - val_loss: 0.8148
Epoch 2/10
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.5906 - val_loss: 0.8145
Epoch 3/10
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.6091 - val_loss: 0.8144
Epoch 4/10
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.6736 - val_loss: 0.8144
Epoch 5/10
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.6175 - val_loss: 0.8144
Epoch 6/10
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.6384 - val_loss: 0.8144
Epoch 7/10
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - loss: 0.7059 - val_loss: 0.8143
Epoch 8/10
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 9s 3ms/step - loss: 0.7600 - val_loss: 0.8137
Epoch 9/10
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 10s 3ms/step - loss: 0.5776 - val_loss: 0.8137
Epoch 10/10
3479/3479 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - loss: 0.6468 - val_loss: 0.8137


In [8]:
# Encode features
X_train_encoded = encoder.predict(X_train)
X_test_encoded = encoder.predict(X_test)

27828/27828 ━━━━━━━━━━━━━━━━━━━━ 32s 1ms/step
6957/6957 ━━━━━━━━━━━━━━━━━━━━ 8s 1ms/step


In [9]:
# XGBoost Classifier
clf = xgb.XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, use_label_encoder=False, eval_metric='logloss')
clf.fit(X_train_encoded, y_train)

C:\Users\Arsalan Khatri\AppData\Local\Programs\Python\Python39\lib\site-packages\xgboost\core.py:158: UserWarning: [14:17:49] WARNING: C:\buildkite-agent\builds\buildkite-windows-cpu-autoscaling-group-i-08cbc0333d8d4aae1-1\xgboost\xgboost-ci-windows\src\learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.1, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=200,
              n_jobs=None, num_parallel_tree=None, random_state=None, ...)

In [10]:
# Predict and evaluate
y_pred = clf.predict(X_test_encoded)
report = classification_report(y_test, y_pred, output_dict=True)
conf_matrix = confusion_matrix(y_test, y_pred)

In [11]:
report, conf_matrix

({'0': {'precision': 0.9816802401884113,
   'recall': 0.98031435983519,
   'f1-score': 0.9809968245698347,
   'support': 111401.0},
  '1': {'precision': 0.9803101178878942,
   'recall': 0.9816762870655086,
   'f1-score': 0.980992726831657,
   'support': 111222.0},
  'accuracy': 0.9809947759216253,
  'macro avg': {'precision': 0.9809951790381528,
   'recall': 0.9809953234503492,
   'f1-score': 0.9809947757007458,
   'support': 222623.0},
  'weighted avg': {'precision': 0.9809957298614994,
   'recall': 0.9809947759216253,
   'f1-score': 0.9809947773481388,
   'support': 222623.0}},
 array([[109208,   2193],
        [  2038, 109184]]))

In [12]:
# Predict and evaluate
y_pred = clf.predict(X_test_encoded)

# Generate classification report
report = classification_report(y_test, y_pred, output_dict=True)

# Confusion matrix
conf_matrix = confusion_matrix(y_test, y_pred)

# Print accuracy
print("Accuracy:", report['accuracy'])

# Agar chahe toh confusion matrix bhi print kar sakte hain
print("Confusion Matrix:\n", conf_matrix)

Accuracy: 0.9809947759216253
Confusion Matrix:
 [[109208   2193]
 [  2038 109184]]


In [13]:
# Save Autoencoder model
autoencoder.save("autoencoder_model.h5")
# Save Encoder model (feature extractor)
encoder.save("encoder_model.h5")

In [14]:
# Save XGBoost model
joblib.dump(clf, "xgboost_model.pkl")

['xgboost_model.pkl']

In [15]:
clf.save_model("xgboost_model.json")

In [16]:
joblib.dump(scaler, "scaler.pkl")

['scaler.pkl']

In [17]:
# Load models
autoencoder.compile(optimizer='adam', loss=MeanSquaredError())
encoder = load_model("encoder_model.h5")
clf = joblib.load("xgboost_model.pkl")
scaler = joblib.load("scaler.pkl")

In [40]:
# Naya data load karo (same format mein)
# new_data = pd.read_csv('../datasets/perfect_Balance_dataset_cic-ids-2017.csv')
i = 0 
for i in range(10):
    new_data = df.sample(7)

# Features nikaalo (label hatao agar ho)
    X_new = new_data.drop(['label', 'label_binary'], axis=1, errors='ignore')  # errors ignore agar label na ho toh

# Scale karo (same scaler jo training mein use kiya tha)
    X_new_scaled = scaler.transform(X_new)

# Encode karo autoencoder encoder se
    X_new_encoded = encoder.predict(X_new_scaled)

# Predict karo XGBoost model se
    predictions = clf.predict(X_new_encoded)

# Predictions: 0 = benign, 1 = attack (jaise aapne label_binary mein define kiya tha)
    print(predictions)


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 59ms/step
[0 0 1 1 1 1 0]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
[1 0 0 1 1 0 1]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
[0 0 0 0 0 0 1]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
[0 0 0 1 0 1 1]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
[1 1 1 1 1 0 0]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
[0 0 1 0 0 0 1]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
[0 0 1 1 0 1 1]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step
[1 0 0 0 1 1 0]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
[1 0 1 0 0 1 0]
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 47ms/step
[1 1 0 1 0 1 0]


In [41]:
from sklearn.metrics import accuracy_score

# Predict on test set
y_pred = clf.predict(X_test_encoded)

# Accuracy calculate karo
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Optional: Classification report aur confusion matrix
from sklearn.metrics import classification_report, confusion_matrix
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))


Accuracy: 0.9809947759216253

Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.98      0.98    111401
           1       0.98      0.98      0.98    111222

    accuracy                           0.98    222623
   macro avg       0.98      0.98      0.98    222623
weighted avg       0.98      0.98      0.98    222623


Confusion Matrix:
 [[109208   2193]
 [  2038 109184]]
